# Demonstration of Florence 2 inference

## 0. Preparation

### Python & Virtual Environment

Before running the script, ensure that Python is installed on your system. The script has been tested with Python 3.11.7 and pip 23.2.1. It is recommended to use a virtual environment to manage dependencies and avoid conflicts with other Python packages on your system.

### Creating a Virtual Environment

To create and activate a virtual environment, follow these steps:

1. **Create the virtual environment**:  
   In the terminal, navigate to the project directory and run:
   ```bash
   python -m venv florenceenv
   ```
   This will create a directory named `florenceenv` in your project directory, which will contain the isolated Python environment.

2. **Activate the virtual environment**:
   - **macOS/Linux**:
     ```bash
     source florenceenv/bin/activate
     ```

   Once activated, the terminal prompt should change to indicate that the virtual environment is active, e.g., `(florenceenv)`.

3. **Install required libraries**:  
   With the virtual environment active, run the following command to install all necessary dependencies:
   ```bash
   pip install -r requirements.txt
   ```

   This will install the required libraries

4. **Deactivate the virtual environment**:  
   After you're done working, you can deactivate the virtual environment by running:
   ```bash
   deactivate
   ```



to Select the interpreter in VS Code :

`Ctrl + Shift + P`

`Python: Select Interpreter`

Choose your venv (it will show something like): 
`./venv/bin/python3.11.2`


## 1. Import required modules:

deimv2 execution scripts are at tools/inference, so we will move to this location

In [ ]:
import os
import torch
from transformers import AutoProcessor, AutoModelForCausalLM
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import csv
import time
import psutil
import argparse
import pandas as pd


functions

In [ ]:

def initialize_model(model,patching,savefigs):
    """Initialize model and processor based on arguments"""
    # Set device and dtype
    device = "cuda:0" if torch.cuda.is_available() else "cpu"
    # torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

    # device = "cpu"
    torch_dtype = torch.float32
    
    # Set model path based on model type
    if model == "base":
        model_path = "./Florence-2-base-ft"
        # model_path = "microsoft/Florence-2-base-ft"
    else:  # large
        model_path = "./Florence-2-large-ft"

    print(f"Using device: {device}")
    print(f"Using model: {model} ({model_path})")
    print(f"Patching: {patching}")
    print(f"Save figures: {savefigs}")

    model = AutoModelForCausalLM.from_pretrained(
        model_path, torch_dtype=torch_dtype, trust_remote_code=True
    ).to(device)

    processor = AutoProcessor.from_pretrained(model_path, trust_remote_code=True)
    print("Model and processor loaded.")
    return model, processor, device, torch_dtype


def create_output_dirs(output_path):
    """Create all necessary output directories"""
    os.makedirs(output_path, exist_ok=True)
    os.makedirs(os.path.join(output_path, "images"), exist_ok=True)
    os.makedirs(os.path.join(output_path, "labels"), exist_ok=True)


def split_image(image, block_size=(640, 640)):
    """Split image into 6 blocks (2 rows x 3 columns)"""
    width, height = image.size
    blocks = []

    # Calculate the step sizes
    x_step = width // 3
    y_step = height // 2

    for i in range(2):  # rows
        for j in range(3):  # columns
            left = j * x_step
            upper = i * y_step
            right = left + x_step
            lower = upper + y_step

            # Ensure we don't go beyond image boundaries
            right = min(right, width)
            lower = min(lower, height)

            block = image.crop((left, upper, right, lower))
            blocks.append((block, (left, upper, right, lower)))

    return blocks


def combine_results(all_results, original_size):
    """Combine results from all blocks into a single result"""
    combined_bboxes = []
    combined_labels = []

    for result, (left, upper, _, _) in all_results:
        if "<OD>" not in result:
            continue

        for bbox, label in zip(result["<OD>"]["bboxes"], result["<OD>"]["labels"]):
            # Adjust coordinates to original image space
            x1, y1, x2, y2 = bbox
            new_bbox = [x1 + left, y1 + upper, x2 + left, y2 + upper]
            combined_bboxes.append(new_bbox)
            combined_labels.append(label)

    return {"<OD>": {"bboxes": combined_bboxes, "labels": combined_labels}}


def process_block(
    block, block_position, prompt, subprompt, processor, model, device, torch_dtype
):
    """Process a single image block"""
    if subprompt:
        prompt = prompt + subprompt
    print(f"Processing block at position {block_position}")

    inputs = processor(text=prompt, images=block, return_tensors="pt").to(
        device, torch_dtype
    )

    generated_ids = model.generate(
        input_ids=inputs["input_ids"],
        pixel_values=inputs["pixel_values"],
        max_new_tokens=1024,
        do_sample=False,
        num_beams=3,
    )

    generated_text = processor.batch_decode(generated_ids, skip_special_tokens=False)[0]
    return processor.post_process_generation(
        generated_text, task="<OD>", image_size=(block.width, block.height)
    )


def process_whole_image_func(
    image, prompt, subprompt, processor, model, device, torch_dtype
):
    """Process the whole image without splitting"""
    if subprompt:
        prompt = prompt + subprompt
    print(f"Processing whole image")

    inputs = processor(text=prompt, images=image, return_tensors="pt").to(
        device, torch_dtype
    )

    generated_ids = model.generate(
        input_ids=inputs["input_ids"],
        pixel_values=inputs["pixel_values"],
        max_new_tokens=1024,
        do_sample=False,
        num_beams=3,
    )

    generated_text = processor.batch_decode(generated_ids, skip_special_tokens=False)[0]
    return processor.post_process_generation(
        generated_text, task="<OD>", image_size=(image.width, image.height)
    )


def plot_and_save_bbox(image, data, output_path, image_name, total_cars, savefigs):
    """Save annotated image with bounding boxes based on savefigs setting"""
    if savefigs == "none":
        return

    fig, ax = plt.subplots()
    ax.imshow(image)

    if savefigs == "all":
        plt.title(f"Cars Detected: {total_cars}", fontsize=8)

    for bbox, label in zip(data["<OD>"]["bboxes"], data["<OD>"]["labels"]):
        if label in ["car", "truck", "van", "vehicle", "bus"]:
            x1, y1, x2, y2 = bbox
            rect = patches.Rectangle(
                (x1, y1), x2 - x1, y2 - y1, linewidth=1, edgecolor="r", facecolor="none"
            )
            ax.add_patch(rect)
            # if savefigs == "all":
            #     plt.text(
            #         x1,
            #         y1,
            #         label,
            #         color="white",
            #         fontsize=4,
            #         bbox=dict(facecolor="red", alpha=0.1),
            #     )

    ax.axis("off")
    plt.savefig(
        f"{output_path}/{image_name}", bbox_inches="tight", pad_inches=0, dpi=300
    )
    plt.close()


def convert_to_yolo_format(bbox, label, img_width, img_height):
    """Convert bbox to YOLO format"""
    x1, y1, x2, y2 = bbox
    center_x = ((x1 + x2) / 2) / img_width
    center_y = ((y1 + y2) / 2) / img_height
    width = (x2 - x1) / img_width
    height = (y2 - y1) / img_height
    class_id = 0  # All vehicles are class 0
    return f"{class_id} {center_x:.6f} {center_y:.6f} {width:.6f} {height:.6f}"


def process_image(
    image_path,
    output_path,
    prompt,
    subprompt,
    processor,
    model,
    device,
    torch_dtype,
    patching,
    savefigs,
):
    """Process a single image"""
    start_time = time.time()
    cpu_before = psutil.cpu_percent()
    mem_before = psutil.virtual_memory().used / (1024 * 1024)
    swap_before = psutil.swap_memory().used / (1024 * 1024)

    original_image = Image.open(image_path)
    original_size = original_image.size
    total_cars = 0

    if not patching:
        # Process the whole image at once
        result = process_whole_image_func(
            original_image, prompt, subprompt, processor, model, device, torch_dtype
        )
        combined_result = {"<OD>": {"bboxes": [], "labels": []}}

        if "<OD>" in result:
            combined_result["<OD>"]["bboxes"] = result["<OD>"]["bboxes"]
            combined_result["<OD>"]["labels"] = result["<OD>"]["labels"]
    else:
        # Split image into blocks
        original_image = original_image.resize((1920, 1280), Image.Resampling.LANCZOS)
        original_size = original_image.size
        print(f"Resized image to: {original_size}")
        blocks = split_image(original_image)

        # Process each block
        all_results = []
        for idx, (block, position) in enumerate(blocks):
            result = process_block(
                block,
                position,
                prompt,
                subprompt,
                processor,
                model,
                device,
                torch_dtype,
            )
            all_results.append((result, position))

        # Combine results
        combined_result = combine_results(all_results, original_size)

    # Get timestamp from filename
    timestamp = os.path.basename(image_path).split(".")[0].split("-", 1)[1]
    image_name = os.path.basename(image_path)

    # Save YOLO format text file
    txt_filename = os.path.splitext(image_name)[0] + ".txt"
    txt_path = os.path.join(output_path, "labels", txt_filename)

    with open(txt_path, "w") as f:
        if "<OD>" in combined_result:
            for bbox, label in zip(
                combined_result["<OD>"]["bboxes"], combined_result["<OD>"]["labels"]
            ):
                # Only save if it's a vehicle (car, truck, etc.)
                if any(
                    vehicle in label.lower()
                    for vehicle in ["car", "truck", "van", "vehicle", "bus"]
                ):
                    yolo_line = convert_to_yolo_format(
                        bbox, label, original_size[0], original_size[1]
                    )
                    f.write(yolo_line + "\n")
                    total_cars += 1
    print(f"Total cars detected: {total_cars}")

    # Save annotated image based on savefigs setting
    if savefigs != "none":
        output_image_path = os.path.join(output_path, "images")
        plot_and_save_bbox(
            original_image,
            combined_result,
            output_image_path,
            image_name,
            total_cars,
            savefigs,
        )

    # Calculate metrics
    processing_time = time.time() - start_time
    cpu_after = psutil.cpu_percent()
    mem_after = psutil.virtual_memory().used / (1024 * 1024)
    swap_after = psutil.swap_memory().used / (1024 * 1024)

    return {
        "image_name": image_name,
        "predicted_cars": total_cars,
        "processing_time": processing_time,
        "cpu_usage": (cpu_before + cpu_after) / 2,
        "memory_used": (mem_before + mem_after) / 2,
        "swap_used": (swap_before + swap_after) / 2,
        "patching": patching,
        "timestamp": timestamp,
    }


def process_directory(
    input_path, output_path, processor, model, device, torch_dtype,patching,
    savefigs
):
    """Process all images in a directory"""
    # Create output directories first
    create_output_dirs(output_path)

    csv_path = os.path.join(output_path, "results.csv")

    with open(csv_path, "w", newline="") as csvfile:
        fieldnames = [
            "image_name",
            "predicted_cars",
            "processing_time",
            "cpu_usage",
            "memory_used",
            "swap_used",
            "patching",
            "timestamp",
        ]
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        writer.writeheader()

        for filename in sorted(os.listdir(input_path)):
            if filename.lower().endswith((".jpg", ".jpeg", ".png")):
                image_path = os.path.join(input_path, filename)
                print(f"Processing {filename}...")

                result = process_image(
                    image_path,
                    output_path,
                    "<OD>",
                    "",
                    processor,
                    model,
                    device,
                    torch_dtype,
                    patching,
                    savefigs,
                )
                if result:
                    writer.writerow(result)


## Inference with SAHI

Import libraries

In [ ]:
# SAHI imports
from sahi.utils.file import download_from_url
from sahi.utils.ultralytics import download_yolo11n_model
from sahi import AutoDetectionModel
from sahi.predict import get_prediction,get_sliced_prediction
from sahi.models.base import DetectionModel
from sahi.prediction import ObjectPrediction


# import os
# import re
# from datetime import datetime, timedelta
# import matplotlib.pyplot as plt
# import matplotlib
from transformers import AutoProcessor, AutoModelForCausalLM
import torch
from typing import List, Optional

# from ultralytics import YOLO
# import time
# import cv2
# import numpy as np
# import json
# import argparse

try:
    import imageio.v3 as iio
except ImportError:
    print("imageio.v3 not found")
from utils_yolo import *
import psutil
import csv
from tqdm import tqdm
import itertools

custom sahi detection model class

In [ ]:

class Florence2DetectionModel(DetectionModel):
    def __init__(
        self,
        model_path: str,
        confidence_threshold: float = 0.3,
        device: str = None,
        load_at_init: bool = True,
    ):
        self.model_path = model_path
        self.confidence_threshold = confidence_threshold
        self._device = device

        # Initialize parent class
        super().__init__(
            model_path=model_path,
            confidence_threshold=confidence_threshold,
            device=device,
            load_at_init=False,
        )

        # Create category mapping for vehicle types
        self.category_name_to_id = {
            "car": 0,
            "suv": 1,
            "truck": 2,
            "van": 3,
            "bus": 4,
            "vehicle": 0,  # map generic "vehicle" to car
            "vehicles": 0,  # map generic "vehicles" to car
            "land vehicle": 0,
            "limousine": 0,
            "minivan": 3,
            "pickup": 2,
            "jeep": 1,
            "sedan": 0,
            "coupe": 0,
            "convertible": 0,
            "wagon": 0,
            "minibus": 4,
        }

        if load_at_init:
            self.load_model()

    def load_model(self):
        try:
            self.processor = AutoProcessor.from_pretrained(
                self.model_path, trust_remote_code=True
            )

            self.model = AutoModelForCausalLM.from_pretrained(
                self.model_path, trust_remote_code=True, torch_dtype=torch.float32
            ).to(self._device)

            print(f"✅ Florence-2 model loaded successfully on {self._device}")

        except Exception as e:
            print(f"❌ Error loading Florence-2 model: {e}")
            raise

        self.category_mapping = None

    def perform_inference(self, image: np.ndarray):
        try:
            if isinstance(image, np.ndarray):
                if image.dtype != np.uint8:
                    image = (image * 255).astype(np.uint8)
                pil_image = Image.fromarray(image)
            else:
                pil_image = image

            prompt = "<OD>"
            inputs = self.processor(
                text=prompt, images=pil_image, return_tensors="pt"
            ).to(self._device)

            inputs["pixel_values"] = inputs["pixel_values"].float()

            print(f"🔍 Performing inference on image size: {pil_image.size}")

            with torch.no_grad():
                generated_ids = self.model.generate(
                    input_ids=inputs["input_ids"],
                    pixel_values=inputs["pixel_values"],
                    max_new_tokens=1024,
                    do_sample=False,
                    num_beams=3,
                )

            generated_text = self.processor.batch_decode(
                generated_ids, skip_special_tokens=False
            )[0]

            print(f"📝 Generated text length: {len(generated_text)} chars")

            self._original_predictions = self.processor.post_process_generation(
                generated_text, task="<OD>", image_size=pil_image.size
            )

        except Exception as e:
            print(f"❌ Error during inference: {e}")
            raise

    def _create_object_prediction_list_from_original_predictions(
        self,
        shift_amount_list: Optional[List[List[int]]] = [[0, 0]],
        full_shape_list: Optional[List[List[int]]] = None,
    ):
        object_prediction_list = []

        if not isinstance(shift_amount_list[0], list):
            shift_amount_list = [shift_amount_list]

        shift_amount = shift_amount_list[0]

        print(f"🔄 Converting predictions with shift: {shift_amount}")

        # Define vehicle classes
        vehicle_classes = {
            "car",
            "suv",
            "truck",
            "van",
            "vehicle",
            "vehicles",
            "land vehicle",
            "limousine",
            "bus",
            "minivan",
            "pickup",
            "jeep",
            "sedan",
            "coupe",
            "convertible",
            "wagon",
            "minibus",
        }

        if self._original_predictions and "<OD>" in self._original_predictions:
            od_predictions = self._original_predictions["<OD>"]

            if "bboxes" in od_predictions and "labels" in od_predictions:
                bboxes = od_predictions["bboxes"]
                labels = od_predictions["labels"]

                print(f"🎯 Processing {len(bboxes)} detections")

                for i, (bbox, label) in enumerate(zip(bboxes, labels)):
                    score = (
                        0.8  # Default score since Florence-2 doesn't provide confidence
                    )
                    label_lower = label.lower().strip()

                    # Check if this is a vehicle
                    is_vehicle = any(
                        vehicle_class in label_lower
                        for vehicle_class in vehicle_classes
                    )

                    if is_vehicle and score >= self.confidence_threshold:
                        # Get category ID from mapping (default to 0 for car if not found)
                        category_id = self.category_name_to_id.get(label_lower, 0)

                        # Apply shift if needed
                        shifted_bbox = [
                            bbox[0] + shift_amount[0],
                            bbox[1] + shift_amount[1],
                            bbox[2] + shift_amount[0],
                            bbox[3] + shift_amount[1],
                        ]

                        # Create ObjectPrediction with both category_id and category_name
                        object_prediction = ObjectPrediction(
                            bbox=shifted_bbox,
                            category_id=category_id,
                            category_name=label,
                            score=score,
                        )
                        object_prediction_list.append(object_prediction)
                        print(f"  ✅ Vehicle {i}: {label} -> ID {category_id}")

        print(f"📊 Total vehicle detections: {len(object_prediction_list)}")
        self._object_prediction_list_per_image = [object_prediction_list]


custom functions

In [ ]:

def plot_side_by_side(
    image1,
    image2,
    output_dir,
    img_name,
    vehicle_count_before,
    vehicle_count_after,
    config_name,
    save_figures=True,
):
    if not save_figures:
        return

    fig, ax = plt.subplots(1, 2, figsize=(12, 6))
    ax[0].imshow(image1)
    ax[0].set_title(f"No SAHI Image: {vehicle_count_before} cars")
    ax[0].axis("off")

    ax[1].imshow(image2)
    ax[1].set_title(f"SAHI {config_name}: {vehicle_count_after} cars")
    ax[1].axis("off")

    plt.tight_layout()
    plt.savefig(f"{output_dir}/comparison_{img_name}_{config_name}.png")
    plt.close()


def calculate_overlap_params(slice_width, slice_height, overlap_ratio):
    """Calculate x_overlap and y_overlap based on slice dimensions and overlap ratio"""
    x_overlap = int(overlap_ratio * slice_width)
    y_overlap = int(overlap_ratio * slice_height)
    return x_overlap, y_overlap


def evaluate_sahi_config(
    detection_model,
    input_image,
    config,
    output_dir,
    img_name,
    # exclude_classes_by_id,
    vehicle_classes,
    save_figures=True,
):
    """Evaluate a single SAHI configuration"""
    start_time = time.time()
    start_metrics = get_system_metrics()

    # Calculate overlaps based on ratio
    x_overlap, y_overlap = calculate_overlap_params(
        config["slice_width"], config["slice_height"], config["overlap_ratio"]
    )

    # Get base prediction without SAHI
    # base_result = get_prediction(input_image, detection_model)
    # vehicle_count_before = sum(1 for pred in base_result.object_prediction_list if pred.category.id in vehicle_class_ids)
    # print(f'config {config["slice_width"]}x{config["slice_height"]} with overlap {config["overlap_ratio"]}')
    # print(f'config {config["postprocess_type"]} with metric {config["postprocess_match_metric"]} and threshold {config["postprocess_match_threshold"]}')
    # Run SAHI prediction
    result = get_sliced_prediction(
        input_image,
        detection_model,
        # slice_height=config["slice_height"],
        # slice_width=config["slice_width"],
        overlap_height_ratio=config["overlap_ratio"],
        overlap_width_ratio=config["overlap_ratio"],
        postprocess_type=config["postprocess_type"],
        postprocess_match_metric=config["postprocess_match_metric"],
        postprocess_match_threshold=config["postprocess_match_threshold"],
        verbose=0,
        # auto_slice_resolution=False,
        # exclude_classes_by_id=exclude_classes_by_id,
    )

    # Calculate metrics
    processing_time = time.time() - start_time
    end_metrics = get_system_metrics()
    avg_metrics = {
        "cpu": (start_metrics["cpu"] + end_metrics["cpu"]) / 2,
        "memory": (start_metrics["memory"] + end_metrics["memory"]) / 2,
        "swap": (start_metrics["swap"] + end_metrics["swap"]) / 2,
    }

    vehicle_count_after = sum(
        1
        for pred in result.object_prediction_list
        if any(
            vehicle_class in pred.category.name.lower()
            for vehicle_class in vehicle_classes
            if vehicle_class != "vehicle registration plate"
        )
    )

    # Save visuals
    config_name = f"{config['slice_width']}x{config['slice_height']}_ov{config['overlap_ratio']}_{config['postprocess_type']}_{config['postprocess_match_metric']}"

    if save_figures:
        result.export_visuals(
            export_dir=output_dir,
            rect_th=2,
            hide_labels=True,
            hide_conf=True,
            file_name=f"result_SAHI_{config_name}_{img_name}",
        )

        # Load images for comparison
        # processed_image = Image.open(f"{output_dir}/result_base_{img_name}.png")
        # processed_image_sahi = Image.open(f"{output_dir}/result_SAHI_{config_name}_{img_name}.png")
        # plot_side_by_side(
        #     processed_image, processed_image_sahi, output_dir,
        #     img_name, vehicle_count_before, vehicle_count_after,
        #     config_name, save_figures
        # )

    return {
        "image_name": img_name,
        "config_name": config_name,
        "predicted_cars": vehicle_count_after,
        "processing_time": processing_time,
        "cpu_usage": avg_metrics["cpu"],
        "memory_used": avg_metrics["memory"],
        "swap_used": avg_metrics["swap"],
        "patching": True,
        "slice_width": config["slice_width"],
        "slice_height": config["slice_height"],
        "overlap_ratio": config["overlap_ratio"],
        "x_overlap": x_overlap,
        "y_overlap": y_overlap,
        "postprocess_type": config["postprocess_type"],
        "postprocess_match_metric": config["postprocess_match_metric"],
        "postprocess_match_threshold": config["postprocess_match_threshold"],
    }


In [ ]:
# DEIM Specific Configs
size="base"
save_figures = False
suffix='sahi'


output_dir = f'demo_output_florence_{suffix}_DEFAULT'
input_dir = "../test_set_FINAL/cam_3"
labels_csv = f'{input_dir}/labels.csv'

In [ ]:
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

model_path = f"./Florence-2-{size}-ft"

detection_model = Florence2DetectionModel(
    model_path=model_path,
    confidence_threshold=0.1,
    # device="cpu",
    device="cuda:0" if torch.cuda.is_available() else "cpu",
)
detection_model

In [ ]:
vehicle_classes = {
    "car",
    "truck",
    "bus",
    "vehicle",
    "limousine",
    "suv",
    "land vehicle",
}  # Florence-2 uses text labels

In [ ]:
# Define hyperparameter search space
configs = []
# '512x512', '768x768', '960x1080 (SAHI Default)', '1920x1080 (Full Frame)'
# Fixed size configurations
# slice_sizes = [
#     (512, 512),
#     (768, 768),
#     (960, 1080),
#     (1080, 960),
# ]  # Added the auto-calculated size
# overlap_ratios = [0, 0.1, 0.25, 0.5, 0.8]

slice_sizes = [
    (960,1080),
]  # Added the auto-calculated size
overlap_ratios = [0.2]
# postprocess_types = ['NMS', 'NMM', 'GREEDYNMM']
# postprocess_metrics = ['IOU', 'IOS']
# thresholds = [0.3, 0.5, 0.7]
postprocess_types = ["GREEDYNMM"]
postprocess_metrics = ["IOS"]
# postprocess_metrics = ["IOU"]

thresholds = [0.5]

In [ ]:
for size in slice_sizes:
    for overlap in overlap_ratios:
        for pp_type in postprocess_types:
            for metric in postprocess_metrics:
                for threshold in thresholds:
                    configs.append(
                        {
                            "slice_width": size[0],
                            "slice_height": size[1],
                            "overlap_ratio": overlap,
                            "postprocess_type": pp_type,
                            "postprocess_match_metric": metric,
                            "postprocess_match_threshold": threshold,
                        }
                    )

# Prepare CSV
csv_file_path = os.path.join(output_dir, "parking_results_hyperparam.csv")
csv_headers = [
    "image_name",
    "config_name",
    "predicted_cars",
    "processing_time",
    "cpu_usage",
    "memory_used",
    "swap_used",
    "patching",
    "slice_width",
    "slice_height",
    "overlap_ratio",
    "x_overlap",
    "y_overlap",
    "postprocess_type",
    "postprocess_match_metric",
    "postprocess_match_threshold",
    "timestamp",
]

with open(csv_file_path, "w", newline="") as csvfile:
    writer = csv.DictWriter(csvfile, fieldnames=csv_headers)
    writer.writeheader()

# Process images
for input_image in tqdm(os.listdir(input_dir)):
    if input_image.lower().endswith((".jpeg", ".jpg", ".png")):
        img_path = os.path.join(input_dir, input_image)
        img_name = os.path.splitext(input_image)[0]
        image_timestamp = extract_timestamp(input_image, mode="filename_reitoria")

        # Get base prediction (no SAHI) only if saving figures
        # if save_figures:
        #     base_result = get_prediction(img_path, detection_model)
        #     base_result.export_visuals(
        #         export_dir=output_dir,
        #         rect_th=2,
        #         hide_labels=False,
        #         hide_conf=True,
        #         file_name=f"result_base_{img_name}"
        #     )
        #     print("saving")
        #     # continue
        # else:
        #     vehicle_count_before = 0  # Will not be used

        # Test all configurations
        for config in tqdm(configs, desc=f"Testing configs for {input_image}"):

            metrics = evaluate_sahi_config(
                detection_model=detection_model,
                input_image=img_path,
                config=config,
                output_dir=output_dir,
                img_name=img_name,
                # exclude_classes_by_id=exclude_classes_by_id,
                vehicle_classes=vehicle_classes,
                save_figures=save_figures,
            )
            metrics["timestamp"] = image_timestamp

            with open(csv_file_path, "a", newline="") as csvfile:
                writer = csv.DictWriter(csvfile, fieldnames=csv_headers)
                writer.writerow(metrics)

In [ ]:
! python3 compute_metrics.py \
    --model "florence_sahi" \
    --parking_metrics "{output_dir}/parking_results_hyperparam.csv" \
    --labels "{labels_csv}" \
    --output_dir "{output_dir}/" \
    --max_spots 15

In [ ]:
print(f"Contents of {output_dir}:")
print(os.listdir(output_dir))

In [ ]:
suffix

In [ ]:

# output_dir = 'demo_output_yolo11m'
summary_path = os.path.join(output_dir, f'summary_metrics_florence_{suffix}.csv')
# output_path
df = pd.read_csv(summary_path)

# List of values to exclude
exclude = ['Average accuracy', 'Balanced accuracy', 'Average precision', 'Average recall', 'Average F1 score']
df_filtered = df[~df.iloc[:, 0].isin(exclude)]

display(df_filtered)

In [ ]:
import glob
show_images = 3
# Get all result images
result_images = glob.glob(os.path.join(output_dir, 'result_SAHI*.png'))

# Display the first image
for img_path in result_images[:show_images]:
    print(f"Displaying: {os.path.basename(img_path)}")
    img = Image.open(img_path)
    display(img)